In [1]:
import pandas as pd

DATA_PATH = "data/amazon_reviews_us_Digital_Video_Games_v1_00.tsv.zip"

df = pd.read_csv(DATA_PATH, sep="\t")
print("Default settings:", df.shape)

Default settings: (144724, 15)


Pandas loaded 144,724 rows and 15 columns.

In [2]:
df = pd.read_csv(DATA_PATH, sep="\t", quoting=3) # 3 = treat quotes as plain text
print("quoting=3:", df.shape)

quoting=3: (145431, 15)


The first version loaded 144,724 rows, and the second version loaded 145,431 rows. No, pandas did not warn me about the difference.
This tells me that I need to manually check my data and row counts because pandas might silently drop or mess up data without giving an error!

In [3]:
peek = df["review_body"].sample(5, random_state=1)
for text in peek:
    print(text, "\n")

I am 16 years old, and I thought it was pure fun.   Plus it's affordable, so I would say it was worth it! 

They are regularly updating this game and the servers seem to always be alive. Great community great game get on and play. 

AMAZING 

My heading says it all. The expansion is very good if you know what you want. It gives you a lot of new cars and car themed house decor with a new rockabilly radio station and some objects thrown in there for good measure. It also adds some new outfits. Most of them are along the car theme but some are fancy suits and dresses. Overall, the content is good. I just wish it didn't cost so much. 

Great game 



One weird thing is the formatting for example, the huge gap of spaces in the middle of the first review ("fun.   Plus"). When you look at the raw data for other reviews, they are filled with HTML tags like <br /> mixed into the text. This comes from the original Amazon website's code . When user pressed "Enter" to create a new paragraph while typing their review, the database saved it as raw HTML line breaks instead of the plain text. 

In [4]:
df["star_rating"].value_counts(normalize=True).sort_index()

star_rating
1    0.171697
2    0.053283
3    0.079962
4    0.140314
5    0.554744
Name: proportion, dtype: float64

The ratings are heavily skewed toward the positive. First, over 55% of all the reviews are 5 star ratings, while 2 star and 3 star ratings make up very little of the data.

This matters for accuracy because if a model guessed "5 stars" for every single review without even reading the text, it would technically be right 55% of the time! A skewed dataset can make a bad model look highly accurate.

In [5]:
df["product_title"].value_counts().head(10)

product_title
Playstation Network Card                                          13642
Xbox Live Subscription                                             7307
Playstation Plus Subscription                                      4712
Xbox Live Gift Card                                                3438
SimCity - Limited Edition                                          3421
Xbox 360 Live Points Card                                          3036
Battlefield 4                                                      1453
Final Fantasy XIV: A Realm Reborn                                  1273
Battlefield 3                                                      1160
Command and Conquer The Ultimate Collection [Online Game Code]      875
Name: count, dtype: int64

The most reviewed items aren't actually video games at all but top four items are all gift cards, network cards, and online subscriptions (like PlayStation Network Cards and Xbox Live Subscriptions).
This is important because reviews for a gift card will probably just be about whether the code worked or not, which is very different from reviewing actual gameplay.

In [6]:
print("Reviews with <br />:", df["review_body"].str.contains("<br />").sum())
df = df.dropna(subset=["review_body"])
df["review_body"] = df["review_body"].str.replace("<br />", " ")
print("After cleaning:", df["review_body"].str.contains("<br />").sum(), "\n")

for text in df.loc[peek.index, "review_body"]:
    print(text, "\n")

Reviews with <br />: 25699
After cleaning: 0 

I am 16 years old, and I thought it was pure fun.   Plus it's affordable, so I would say it was worth it! 

They are regularly updating this game and the servers seem to always be alive. Great community great game get on and play. 

AMAZING 

My heading says it all. The expansion is very good if you know what you want. It gives you a lot of new cars and car themed house decor with a new rockabilly radio station and some objects thrown in there for good measure. It also adds some new outfits. Most of them are along the car theme but some are fancy suits and dresses. Overall, the content is good. I just wish it didn't cost so much. 

Great game 



There were 25,699 reviews with line breaks before cleaning, and 0 after.
The huge gaps of spaces are still there (like in the first review). However, this probably won't change how a model reads it, because models usually ignore extra whitespace and just focus on the words themselves (like "fun" and "affordable").

In [7]:
sample = df .groupby("star_rating").sample(n=100, random_state=42)
complaints = df[df["star_rating"] == 1].sample(n=20, random_state=42)
print(len(sample), len(complaints))

500 20


We take the same number of each star rating to balance the dataset. As we saw earlier, over 55% of the reviews are 5 star. If we just took 500 random reviews, we wouldn't have enough negative reviews to properly test if the model actually understands bad sentiment.

In [8]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)
print(sample["star_rating"].iloc[0], sample["review_body"].iloc[0][:200])
classifier(sample["review_body"].iloc[0])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

1 Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get 


[{'label': 'NEGATIVE', 'score': 0.9998071789741516}]

The keys in the result are 'label' and 'score'. The score represents the model's confidence level (it is 99.98% sure).
Yes, the model agrees with the customer! The customer gave it 1 star, and the model predicted NEGATIVE.

In [9]:
results = classifier(sample["review_body"].tolist(), truncation=True)
sample["label"] = [r["label"] for r in results]
sample["score"] = [r["score"] for r in results]
sample[["star_rating", "label", "score"]].head(10)

,star_rating,label,score
40836,1,NEGATIVE,0.999807
85497,1,NEGATIVE,0.990978
124434,1,NEGATIVE,0.998919
118528,1,NEGATIVE,0.999704
13480,1,NEGATIVE,0.999466
47179,1,NEGATIVE,0.999234
130193,1,NEGATIVE,0.999613
126902,1,NEGATIVE,0.999815
58985,1,NEGATIVE,0.999799
22608,1,NEGATIVE,0.999736


It took about 2-3 minutes to run 500 reviews. Since 145,000 is 290 times bigger than 500, it would take roughly 725 minutes (over 10 hours) to run the whole dataset.
truncation=True protects the model from crashing. These AI models have a strict maximum word limit they can process at one time. If a review has 5,000 words, truncation just chops off the extra text at the end so it doesn't break the model.

In [10]:
def stars_to_sentiment(stars):
    """Convert a star rating into the sentiment the customer expressed."""
    if stars <= 2:
        return "NEGATIVE"
    if stars >= 4:
        return "POSITIVE"
    return "NEUTRAL"

sample["truth"] = sample["star_rating"].apply(stars_to_sentiment)
graded = sample[sample["truth"] != "NEUTRAL"]
print("Accuracy:", (graded["label"] == graded["truth"]).mean())

Accuracy: 0.8775


We leave out the 3 star reviews because the model only predicts "POSITIVE" or "NEGATIVE". A 3 star review is usually mixed or neutral, so it wouldn't be fair to grade the model on those since it doesn't have a "NEUTRAL" option to pick from.

The accuracy is 87.75%. I would trust this model to get a general overview of customer sentiment, but I might not trust it to fully automate important customer service decisions without human review, since it still gets 1 out of 10 wrong.

In [11]:
pd.crosstab(sample["star_rating"], sample["label"])

label,NEGATIVE,POSITIVE
star_rating,,
1,96,4
2,95,5
3,66,34
4,23,77
5,17,83


The model gets 4 star reviews wrong the most often. 
It incorrectly labeled 23 of the 4 star reviews as "NEGATIVE".

Since the model only knows POSITIVE and NEGATIVE, it was forced to split the 3 star reviews into those two buckets. It labeled 66 of them as NEGATIVE and 34 of them as POSITIVE, leaning slightly negative overall.

In [12]:
mistakes = graded[graded["label"] != graded["truth"]]
for review in mistakes.head(5).itertuples():
    print(review.star_rating, review.label, round(review.score, 2))
    print(review.review_body[:300], "\n")

1 POSITIVE 0.99
Don't get me wrong, the game concept is awesome. Graphics are great.  BUT  If you don't want to lose your progress when the Simcity Servers stop responding, don't buy this. There is no option to save locally just saved to their server so the smallest error will lose your work.  Overall it's a good g 

1 POSITIVE 1.0
Enjoys game 

1 POSITIVE 0.73
I wasted $7. im sure its a good game but i cant even get to the game part. i tried. a lot. 

1 POSITIVE 1.0
Dose not work right on PSVITA so was not happy but it is a fun game love twisted metal games. 

2 POSITIVE 0.87
I bought this game for my 8 year old daughter who likes the kingdom of keflings game on xbox. The graphics are very simple and it is a basic strategy/building style game. Unfortunately, it looks like it would have been cutting edge 15 to 20 years ago, but it really didn't entertain my kids for more  



In the second review ("Enjoys game"), the customer's rating was definitely misleading—they wrote a positive comment but accidentally gave it 1 star!
For the others, the model got confused. It saw strong positive words ("awesome", "great", "good game", "love") and stopped paying attention to the context, completely missing the complaints about broken servers or wasted money.

In [ ]:
# Load the 1-to-5 star prediction model
star_classifier = pipeline(
    "sentiment-analysis", 
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

# Run the model on our 500 reviews
star_results = star_classifier(sample["review_body"].tolist(), truncation=True)

# The model outputs labels like "4 stars", so we grab just the first character (the number)
sample["predicted_stars"] = [int(r["label"][0]) for r in star_results]

# Calculate how accurate it is
exact_match = (sample["predicted_stars"] == sample["star_rating"]).mean()
within_one = (abs(sample["predicted_stars"] - sample["star_rating"]) <= 1).mean()

print("Exact match:", exact_match)
print("Within one star:", within_one)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

It gets the exact star rating 57.4% of the time, and it is within one star 91.2% of the time.
If I were building a dashboard, I would probably use the simple Positive/Negative model from Part 2. It is faster, less complicated, and highly accurate (around 87%) for telling us if people are generally happy or mad. Predicting exact stars is much harder for the AI, and 57% exact accuracy might be too low if the publisher wants precise ratings.

In [ ]:
# 1. Set up the model
zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# 2. Define the categories (Check step-15.png on Canvas for the exact words!)
labels = ["price", "bugs", "gameplay", "customer service"] 

# 3. Run the model on your 20 complaint reviews
results = zero_shot(complaints["review_body"].tolist(), labels)

# 4. Save the top predicted topic for each review
complaints["topic"] = [r["labels"][0] for r in results]

# 5. Print out the count of what people complain about most
print(complaints["topic"].value_counts(), "\n")

# 6. Print three of the reviews with their topics to see if they make sense
for row in complaints.head(3).itertuples():
    print(f"Topic: {row.topic}")
    print(row.review_body[:300], "\n")

Customers complain about "gameplay" the most (10 out of the 20 reviews).
The model's choices mostly make sense. The first review specifically mentions "$30 extra", so "price" is perfect. The third review complains about a slow pace, which fits "gameplay". 

However, the second review is about the game not working on their PC despite meeting requirements that should probably be "bugs" instead of "gameplay".

In [ ]:
more_labels = labels + ["technical problems", "HR policy"]
new_results = zero_shot(complaints["review_body"].head(5).tolist(), candidate_labels=more_labels)

for old, new in zip(results[:5], new_results):
    print("Before:", old["labels"][0], round(old["scores"][0], 2))
    print("After: ", new["labels"][0], round(new["scores"][0], 2), "\n")

For the first three reviews, the top label stayed the same, but the confidence score dropped (example: gameplay went from 0.73 to 0.41). For the last two, the label completely changed to the new "technical problems" category with a high score.

Since scores must add up to 1, adding more choices dilutes the probability. The model has to split its 100% confidence across more options, which is why the scores for the original labels went down even when they still won.

This shows that for a real product, you must choose your labels very carefully. If you have overlapping labels (like "bugs" and "technical problems"), the model gets confused and splits the score between them. You need clear, different categories.